# Walmart Weekly Sales — Random Forest Regressor

This notebook trains a **Random Forest Regressor** to predict weekly sales across 45 Walmart stores using the [Walmart Dataset](https://www.kaggle.com/datasets/yasserh/walmart-dataset) from Kaggle.

**Pipeline overview:**
1. Load & inspect data
2. Feature engineering (date decomposition, encoding)
3. Feature selection (carry over from linear regression notebook)
4. Train / test split + log-transform target
5. Train a Random Forest model with GridSearchCV hyperparameter tuning
6. Evaluate with R², MAE, and RMSE
7. Interpret feature importances and OOB score
8. Diagnostic plots and learning curve
9. Compare against Linear Regression baseline

## 1. Setup

In [ ]:
!pip install kagglehub -q

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("Libraries loaded.")

## 2. Load Data

In [ ]:
import kagglehub

path = kagglehub.dataset_download('yasserh/walmart-dataset')
file_path = os.path.join(path, os.listdir(path)[0])

# --- Alternative: upload Walmart.csv manually and set ---
# file_path = 'Walmart.csv'

df = pd.read_csv(file_path)
print(f'Shape: {df.shape}')
df.head()

## 3. Feature Engineering

Same feature engineering as the linear regression notebook for consistency and fair comparison:
- Date decomposed into Week, Month, Year
- Holiday_Flag label-encoded (already binary — no-op)
- Fuel_Price dropped (Pearson p = 0.4478, not significant)
- Store one-hot encoded with drop_first=True

In [ ]:
df['Date']  = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year
df.drop(columns=['Date'], inplace=True)

le = LabelEncoder()
df['Holiday_Flag'] = le.fit_transform(df['Holiday_Flag'])

print("Dtypes after feature engineering:")
print(df.dtypes)

## 4. Target Log-Transformation

`Weekly_Sales` is right-skewed (range: ~$210k–$3.8M). Log-transforming the target:
- Reduces influence of extreme outlier stores and holiday peaks
- Aligns with the Kaggle RMSE-on-log-scale evaluation metric
- Allows fair comparison against published benchmarks

We train and evaluate on log(Weekly_Sales), then convert predictions back to dollars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Weekly_Sales'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly_Sales (original scale)', fontsize=12)
axes[0].set_xlabel('Weekly Sales ($)')
axes[0].set_ylabel('Frequency')

log_sales = np.log(df['Weekly_Sales'])
axes[1].hist(log_sales, bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('log(Weekly_Sales)', fontsize=12)
axes[1].set_xlabel('log(Weekly Sales)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution: Before and After Log-Transform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Original skewness:         {df['Weekly_Sales'].skew():.4f}")
print(f"Log-transformed skewness:  {log_sales.skew():.4f}")

## 5. Feature Selection & Encoding

In [ ]:
target = 'Weekly_Sales'

# Drop Fuel_Price (not statistically significant)
df.drop(columns=['Fuel_Price'], inplace=True)

# One-hot encode Store
df_model = pd.get_dummies(df, columns=['Store'], drop_first=True)

X = df_model.drop(columns=[target])
y = np.log(df_model[target])   # log-transform target

print(f"Feature matrix shape: {X.shape}")
print(f"Sample features: {X.columns.tolist()[:10]} ...")

## 6. Train / Test Split

80/20 split with fixed random_state=42 for reproducibility and consistency across notebooks.

**Note:** Random Forest does not require feature scaling — decision trees split on thresholds
and are scale-invariant. We train directly on the raw feature values.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

## 7. Random Forest — Hyperparameter Tuning

Random Forest builds many decision trees independently on bootstrap samples of the training data
and averages their predictions. Key hyperparameters:

| Parameter | Description |
|---|---|
| `n_estimators` | Number of trees. More trees = lower variance, but diminishing returns beyond ~300. |
| `max_features` | Number of features considered at each split. Lower = more diverse trees, reduces correlation. Typical values: sqrt(n_features) for classification, n_features/3 for regression. |
| `max_depth` | Maximum depth of each tree. None = fully grown (may overfit). |
| `min_samples_split` | Minimum samples required to split an internal node. Higher = simpler trees. |

GridSearchCV with 5-fold cross-validation. OOB (Out-of-Bag) score is also enabled as a free
internal validation estimate — each tree is evaluated on the ~37% of training samples not used
in its bootstrap sample.

In [ ]:
rf = RandomForestRegressor(
    random_state = 42,
    oob_score    = True,
    n_jobs       = -1
)

n_features = X_train.shape[1]

param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_features'    : ['sqrt', 'log2', max(1, n_features // 3)],
    'max_depth'       : [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    estimator  = rf,
    param_grid = param_grid,
    scoring    = 'r2',
    cv         = 5,
    n_jobs     = -1,
    verbose    = 1
)

rf_grid.fit(X_train, y_train)

print("\nBest parameters:")
for k, v in rf_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV R²: {rf_grid.best_score_:.4f}")
print(f"OOB R²:     {rf_grid.best_estimator_.oob_score_:.4f}")

## 8. Evaluate on Test Set

In [ ]:
best_rf = rf_grid.best_estimator_

y_pred_train_log = best_rf.predict(X_train)
y_pred_test_log  = best_rf.predict(X_test)

# Metrics on log scale (Kaggle-comparable)
train_r2_log  = r2_score(y_train, y_pred_train_log)
test_r2_log   = r2_score(y_test,  y_pred_test_log)
test_rmse_log = mean_squared_error(y_test, y_pred_test_log) ** 0.5

# Convert back to original scale
y_pred_test_orig  = np.exp(y_pred_test_log)
y_test_orig       = np.exp(y_test)
y_pred_train_orig = np.exp(y_pred_train_log)
y_train_orig      = np.exp(y_train)

train_r2  = r2_score(y_train_orig, y_pred_train_orig)
test_r2   = r2_score(y_test_orig,  y_pred_test_orig)
test_mae  = mean_absolute_error(y_test_orig, y_pred_test_orig)
test_rmse = mean_squared_error(y_test_orig,  y_pred_test_orig) ** 0.5

print('=' * 50)
print('RANDOM FOREST — Results (Log Scale)')
print('=' * 50)
print(f'  Train R²:         {train_r2_log:.4f}')
print(f'  Test  R²:         {test_r2_log:.4f}')
print(f'  Test  RMSE (log): {test_rmse_log:.4f}')
print(f'  OOB   R²:         {best_rf.oob_score_:.4f}')
print()
print('=' * 50)
print('RANDOM FOREST — Results (Original Scale)')
print('=' * 50)
print(f'  Train R²:   {train_r2:.4f}')
print(f'  Test  R²:   {test_r2:.4f}')
print(f'  Test  MAE:  ${test_mae:,.0f}')
print(f'  Test  RMSE: ${test_rmse:,.0f}')

## 9. Overfitting Check

In [ ]:
gap = train_r2_log - test_r2_log
print(f"Train R² (log): {train_r2_log:.4f}")
print(f"Test  R² (log): {test_r2_log:.4f}")
print(f"OOB   R²:       {best_rf.oob_score_:.4f}")
print(f"Gap (train-test): {gap:.4f}")
if gap < 0.02:
    print("-> No significant overfitting.")
elif gap < 0.05:
    print("-> Slight overfitting. Consider increasing min_samples_split or reducing max_depth.")
else:
    print("-> Overfitting detected. Reduce model complexity.")

## 10. Feature Importance

In [ ]:
importances = best_rf.feature_importances_
feat_df = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top20 = feat_df.head(20)

plt.figure(figsize=(10, 7))
sns.barplot(data=top20, x='Importance', y='Feature', palette='Greens_r')
plt.title('Random Forest — Top 20 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(top20.head(10).to_string(index=False))

## 11. Diagnostic Plots

**Residuals vs Predicted (log scale):** checks homoscedasticity.
**Actual vs Predicted (original scale):** shows prediction accuracy across the sales range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
residuals = y_test.values - y_pred_test_log
axes[0].scatter(y_pred_test_log, residuals, alpha=0.3, s=10, color='seagreen')
axes[0].axhline(0, color='crimson', linewidth=1.5)
axes[0].set_xlabel('Predicted (log scale)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted (log scale)')

# Actual vs Predicted
axes[1].scatter(y_test_orig, y_pred_test_orig, alpha=0.3, s=10, color='seagreen')
lims = [min(y_test_orig.min(), y_pred_test_orig.min()),
        max(y_test_orig.max(), y_pred_test_orig.max())]
axes[1].plot(lims, lims, color='crimson', linewidth=1.5)
axes[1].set_xlabel('Actual Weekly Sales ($)')
axes[1].set_ylabel('Predicted Weekly Sales ($)')
axes[1].set_title(f'Actual vs Predicted  (R² = {test_r2:.3f})')

plt.suptitle('Random Forest — Diagnostic Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Learning Curve

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_rf, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='r2', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color='seagreen', label='Training R²')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='seagreen')
plt.plot(train_sizes, val_mean, 'o-', color='darkorange', label='Validation R²')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='darkorange')
plt.xlabel('Training Set Size')
plt.ylabel('R²')
plt.title('Learning Curve — Random Forest', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final training R²:   {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Final validation R²: {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")

## 13. Comparison Against Linear Regression Baseline

In [ ]:
# Retrain Linear Regression on same split for fair comparison
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr_log  = lr.predict(X_test_sc)
y_pred_lr_orig = np.exp(y_pred_lr_log)

lr_r2_log   = r2_score(y_test, y_pred_lr_log)
lr_rmse_log = mean_squared_error(y_test, y_pred_lr_log) ** 0.5
lr_r2       = r2_score(y_test_orig, y_pred_lr_orig)
lr_mae      = mean_absolute_error(y_test_orig, y_pred_lr_orig)
lr_rmse     = mean_squared_error(y_test_orig, y_pred_lr_orig) ** 0.5

comparison = pd.DataFrame({
    'Model'         : ['Linear Regression (baseline)', 'Random Forest'],
    'Test R² (log)' : [round(lr_r2_log, 4),   round(test_r2_log, 4)],
    'RMSE (log)'    : [round(lr_rmse_log, 4),  round(test_rmse_log, 4)],
    'Test R²'       : [round(lr_r2, 4),        round(test_r2, 4)],
    'Test MAE ($)'  : [f'${lr_mae:,.0f}',      f'${test_mae:,.0f}'],
    'Test RMSE ($)' : [f'${lr_rmse:,.0f}',     f'${test_rmse:,.0f}']
})

print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Linear Regression', 'Random Forest']
colors = ['#4878CF', '#3A9E5E']

for ax, metric, vals in zip(
    axes,
    ['Test R² (log)', 'RMSE (log)'],
    [[lr_r2_log, test_r2_log], [lr_rmse_log, test_rmse_log]]
):
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Linear Regression vs Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()